# 第 1 周末练习 —— 抓取网站并给出 UX 改进建议

## 练习目标（理念）

为了展示你对 **浏览器自动化抓取** 与 **LLM 分析** 的熟悉程度，请构建一个小工具：

- **输入**：一个网页 URL
- **过程**：用 Playwright 无头浏览器打开页面，再用 BeautifulSoup 清洗 HTML，抽出可读文本
- **输出**：把标题 + 正文片段交给本地/兼容接口上的小模型，生成 **UX（用户体验）改进建议**

这是课程里「网站理解 / 内容提取」主题的延伸：先可靠拿到页面文本，再让模型做分析。

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | Playwright `chromium.launch` + `page.goto` |
| HTML 清洗 | BeautifulSoup 去掉 script/style 等标签 |
| Chat Completions | `openai.chat.completions.create(...)` |
| 异步编程 | `async` / `await` / `asyncio.run` |

## 怎么跑

1. 已安装并可用 Playwright Chromium（以及本机可访问的模型端点；原代码直接调 `openai.chat.completions`）
2. 运行下面代码单元格；如需换站，改 `main()` 里的 `url`
3. 终端/输出区会打印页面标题与模型给出的 Suggestions


In [ ]:
# ========== 导入：异步运行时 + 浏览器抓取 + HTML 解析 + OpenAI ==========

# 导入 asyncio：用事件循环跑异步的 Playwright 与 main()
import asyncio
# 导入 sys：标准库占位/环境相关（本文件主路径未必直接用到，保持原导入）
import sys
# 导入 openai 模块：后面用 openai.chat.completions.create 调聊天接口
import openai
# 从 playwright.async_api 导入 async_playwright：异步启动浏览器、打开页面
from playwright.async_api import async_playwright
# 从 bs4 导入 BeautifulSoup：解析 HTML、删除噪声标签、提取正文文本
from bs4 import BeautifulSoup

class WebsiteAnalyzer:
    """给定 URL：抓取页面 → 清洗文本 → 调用模型生成 UX 建议。"""

    def __init__(self, url):
        # 要分析的目标网址（字符串保持原样，不翻译 URL）
        self.url = url
        # 页面标题，initialize() 成功后填入
        self.title = ""
        # 清洗后的正文文本
        self.text = ""
        # 模型返回的改进建议
        self.suggestions = ""

    async def initialize(self):
        # 异步上下文：启动 Playwright，退出时自动清理驱动资源
        async with async_playwright() as p:
            # 启动无头 Chromium（headless=True：不弹出可见窗口，适合服务器/笔记本）
            browser = await p.chromium.launch(headless=True)
            # 新建浏览器上下文：可设置 UA、视口，降低「一眼假爬虫」被拦的概率
            context = await browser.new_context(
                user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
                viewport={'width': 1920, 'height': 1080},
            )
            # 在该上下文中打开一个新标签页
            page = await context.new_page()
            try:
                # 导航到目标 URL；timeout 单位毫秒，页面慢时可调大（此处保持 90000）
                await page.goto(self.url, timeout=90000)
                # 等待网络大致空闲，尽量让动态内容加载完再取 HTML
                await page.wait_for_load_state('networkidle', timeout=30000)
                # 读取浏览器里的文档标题
                self.title = await page.title()
                # 取出渲染后的完整 HTML 字符串
                content = await page.content()
                # 用 BeautifulSoup 解析 HTML
                soup = BeautifulSoup(content, 'html.parser')
                # 删除脚本、样式、图片、输入框等对「读正文」噪声大的标签
                for tag in soup.find_all(["script", "style", "img", "input"]):
                    tag.decompose()
                # 从 body 抽文本：换行分隔并 strip；若无 body 则空字符串
                self.text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
                # 把清洗后的文本交给模型，生成 UX 建议并存到属性里
                self.suggestions = self.get_suggestions()
            finally:
                # 无论成功失败都关闭浏览器，避免僵尸进程占资源
                await browser.close()

    def get_suggestions(self):
        # 调用 Chat Completions：model / prompt 字符串保持原样（改译会改变行为）
        response = openai.chat.completions.create(
            model="llama3.2:1b",
            messages=[{"role": "user", "content": f"Analyze this website and give UX suggestions.\nTitle: {self.title}\nContent: {self.text[:4000]}"}]
        )
        # 返回助手消息正文
        return response.choices[0].message.content

async def main():
    # 示例文章 URL：可换成你想分析的页面（保持原链接不改）
    url = "https://medium.com/@copy.kolary/10-simple-and-effective-ways-to-develop-creativity-65faf94bd442"
    # 创建分析器实例
    analyzer = WebsiteAnalyzer(url)
    # 异步执行：打开浏览器、抓取、清洗、调模型
    await analyzer.initialize()
    # 打印标题与建议，方便在笔记本输出区阅读
    print("\nTitle:\n", analyzer.title)
    print("\nSuggestions:\n", analyzer.suggestions)

# 脚本入口：直接运行该文件/单元格时，用 asyncio.run 启动异步 main
if __name__ == "__main__":
    asyncio.run(main())
